<a href="https://colab.research.google.com/github/harshchamp/ProteinExpression/blob/main/Harsh_Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import  pandas as pd
import duckdb as db

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving customers.csv to customers (4).csv
Saving fx_rates.csv to fx_rates (4).csv
Saving payments.csv to payments (4).csv
Saving refunds.csv to refunds (4).csv


In [ ]:
df_Payment = pd.read_csv("/content/payments.csv")
df_customer = pd.read_csv("/content/customers.csv")
df_fx_rates= pd.read_csv("/content/fx_rates.csv")
df_refunds= pd.read_csv("/content/refunds.csv")

In [ ]:
df_Payment

,id,customer_id,created_at_utc,amount_minor,currency,status,updated_at_utc
0,P0001,C001,2025-07-01T09:05:00Z,2500,EUR,posted,2025-07-01T09:10:00Z
1,P0002,C002,2025-07-01T10:15:00Z,1500,GBP,posted,2025-07-01T10:16:00Z
2,P0003,C003,2025-07-02T11:00:00Z,1200,USD,posted,2025-07-02T11:05:00Z
3,P0004,C004,2025-07-02T14:20:00Z,2200,EUR,posted,2025-07-02T14:25:00Z
4,P0005,C003,2025-07-03T09:45:00Z,3000,USD,posted,2025-07-03T09:50:00Z
5,P0006,C002,2025-07-03T13:05:00Z,2000,GBP,posted,2025-07-03T13:06:00Z
6,P0007,C005,2025-07-04T08:30:00Z,1800,USD,posted,2025-07-04T08:31:00Z
7,P0008,C006,2025-07-04T16:10:00Z,1600,EUR,cancelled,2025-07-04T16:15:00Z
8,P0009,C001,2025-07-05T10:00:00Z,2400,EUR,posted,2025-07-05T10:02:00Z
9,P0010,C002,2025-07-05T11:20:00Z,1300,GBP,posted,2025-07-05T11:25:00Z


In [ ]:
df_customer

,customer_id,country
0,C001,DE
1,C002,GB
2,C003,US
3,C004,FR
4,C005,ES
5,C006,IE


In [ ]:
df_refunds

,id,original_payment_id,created_at_utc,amount_minor,currency,status,updated_at_utc
0,R1001,P0002,2025-07-02T09:00:00Z,1500,GBP,refunded,2025-07-02T09:05:00Z
1,R1002,P0005,2025-07-04T10:00:00Z,1000,USD,refunded,2025-07-04T10:01:00Z
2,R1003,P0007,2025-07-05T09:30:00Z,200,USD,refunded,2025-07-05T09:31:00Z
3,R1003,P0007,2025-07-05T09:30:00Z,200,USD,refunded,2025-07-05T09:30:30Z
4,R1004,P0011,2025-07-06T16:00:00Z,600,GBP,refunded,2025-07-06T16:05:00Z


In [ ]:
df_fx_rates

,date_utc,base_ccy,quote_ccy,rate
0,2025-07-01,USD,EUR,0.920
1,2025-07-01,GBP,EUR,1.170
2,2025-07-01,EUR,EUR,1.000
3,2025-07-02,USD,EUR,0.921
4,2025-07-02,GBP,EUR,1.169
5,2025-07-02,EUR,EUR,1.000
6,2025-07-03,USD,EUR,0.923
7,2025-07-03,GBP,EUR,1.168
8,2025-07-03,EUR,EUR,1.000
9,2025-07-04,USD,EUR,0.924


In [ ]:
#Initating DuckDB
con = db.connect(database=':memory:')

In [ ]:
#Task to clean Payments data
con.execute("""
-- Create or replace the 'payments' table from the raw 'df_Payment' DataFrame.
-- This table is cleaned by filtering for 'posted' statuses and de-duplicating
-- to keep only the latest record for each payment ID based on 'updated_at_utc'.
CREATE OR REPLACE TABLE payments AS SELECT * FROM df_Payment
WHERE status = 'posted'
QUALIFY row_number() OVER (
    PARTITION BY id
    ORDER BY id ASC , updated_at_utc DESC
) = 1""")

In [ ]:
#Task to clean Refund Table
con.execute("""CREATE OR REPLACE TABLE refunds AS SELECT * FROM df_refunds
-- De-duplicate refunds to keep only the latest record for each original payment ID
-- based on the most recent 'updated_at_utc' timestamp.
QUALIFY row_number() OVER (
    PARTITION BY original_payment_id
    ORDER BY  updated_at_utc DESC
) = 1 """)

In [ ]:
#Creating Tables for Customers and Fx-Rates
con.execute("CREATE OR REPLACE TABLE customers AS SELECT * FROM df_customer")
con.execute("CREATE OR REPLACE TABLE fx_rates AS SELECT * FROM df_fx_rates")

In [ ]:
#Defining Rules library to check data sanity ~Added few extra checks
rules = {
    "not_null: payments.id":
        "SELECT COUNT(*) FROM payments WHERE id IS NULL",

    "uniqueness: payments.id":
        "SELECT COUNT(*) FROM (SELECT id FROM payments GROUP BY id HAVING COUNT(*) > 1)",

    "accepted_values: payments.status":
        "SELECT COUNT(*) FROM payments WHERE status NOT IN ('posted', 'cancelled', 'refunded')",

    "referential_integrity: payments.customer_id":
        """SELECT COUNT(*) FROM payments p
           LEFT JOIN customers c ON p.customer_id = c.customer_id
           WHERE c.customer_id IS NULL""",
    "Valid_Amount: payments.amount": # checked if amount is in negative
        "SELECT COUNT(*) FROM payments WHERE amount_minor <= 0",
    "Valid_Refund_amount: refunds.amount":
        "SELECT COUNT(*) FROM refunds WHERE amount_minor <= 0",
   "Valid_Customer: customers.customer_id": # checked if there is duplicate customer with same country(considering multiple nationality case as well)
   "SELECT COUNT(*) FROM (SELECT customer_id FROM customers GROUP BY customer_id, country HAVING COUNT(*) > 1)",
    "refund_integrity: original_payment_exists":
        """SELECT COUNT(*) FROM refunds r
           LEFT JOIN payments p ON r.original_payment_id = p.id
           WHERE p.id IS NULL"""
}

In [ ]:
def run_data_controls(rules_dict, connection):
    """
    Executes a set of data quality rules against the raw data and reports the results.
    Rules are expected to return a count of violations (0 for no violations).
    """
    results = []
    passed_count = 0

    for test_name, query in rules_dict.items():
        # Execute the query to get the number of violations
        error_count = connection.execute(query).fetchone()[0]

        if error_count == 0:
            passed_count += 1
            results.append(f" PASSED: {test_name}")
        else:
            results.append(f" FAILED: {test_name} ({error_count} violations found)")

    # Print a summary report of all checks
    print("--- RAW DATA CONTROL REPORT ---")
    print(f"{len(rules_dict)} checks run - {passed_count} passed")
    print("\nOutput:")
    for line in results:
        print(f"  {line}")

    # Return True if all checks passed, False otherwise
    return passed_count == len(rules_dict)

# Usage example: run the data controls
all_passed = run_data_controls(rules, con)

--- RAW DATA CONTROL REPORT ---
8 checks run - 8 passed

Output:
   PASSED: not_null: payments.id
   PASSED: uniqueness: payments.id
   PASSED: accepted_values: payments.status
   PASSED: referential_integrity: payments.customer_id
   PASSED: Valid_Amount: payments.amount
   PASSED: Valid_Refund_amount: refunds.amount
   PASSED: Valid_Customer: customers.customer_id
   PASSED: refund_integrity: original_payment_exists


Below code box execute the creation of fact daily revenu table with check of 1 day prior date at max to match FX Rate.

In [ ]:
con.execute("""

-- Create a new table 'fact_daily_revenue_eur' to store aggregated daily revenue in EUR.
CREATE OR REPLACE TABLE fact_daily_revenue_eur AS (
WITH
-- CTE 1: cleaned_payments
-- This CTE prepares the payments data by ensuring only valid, posted payments are considered.
cleaned_payments AS (
  SELECT * FROM payments
),
-- CTE 2: Payment_EUR
-- Converts payment amounts to EUR using available FX rates.
-- It applies a 1-day prior fallback for FX rates if an exact date match is not found.
-- Payments without a suitable FX rate within this window will have a gross_amount_eur of 0.
Payment_EUR AS (
    SELECT
        cp.*,
        -- Calculate gross amount in EUR.
        -- If an FX rate exists and is within 1 day of the payment date, apply the conversion.
        -- Otherwise, the gross amount in EUR is 0, indicating a missing or unsuitable FX rate.
        CASE
            WHEN fr.rate IS NOT NULL AND (DATE(cp.created_at_utc) - DATE(fr.date_utc)) <= 1
            THEN cp.amount_minor * fr.rate
            ELSE 0
        END as gross_amount_eur,
        -- Flag payments where no suitable FX rate was found for conversion. This flag is not used in the final aggregation.
        CASE
            WHEN fr.rate IS NULL OR (DATE(cp.created_at_utc) - DATE(fr.date_utc)) > 1
            THEN TRUE
            ELSE FALSE
        END as missing_fx
    FROM cleaned_payments cp
    LEFT JOIN fx_rates fr
        ON cp.currency = fr.base_ccy
        AND DATE(fr.date_utc) <= DATE(cp.created_at_utc)
    -- Select the most recent FX rate for each payment if multiple rates are available within the window.
    QUALIFY row_number() OVER (
        PARTITION BY cp.id
        ORDER BY fr.date_utc DESC
    ) = 1
),
-- CTE 3: cleaned_refunds
-- This CTE prepares the refunds data.
cleaned_refunds AS (
    SELECT * FROM refunds
),
-- CTE 4: Refunds_EUR
-- Converts refund amounts to EUR using available FX rates, similar to payments.
Refunds_EUR AS (
    SELECT
        cr.*,
        -- Calculate refund amount in EUR.
        -- Applies the same 1-day prior fallback logic as for payments.
        CASE
            WHEN fr.rate IS NOT NULL AND (DATE(cr.created_at_utc) - DATE(fr.date_utc)) <= 1
            THEN cr.amount_minor * fr.rate
            ELSE 0
        END as refunds_eur
    FROM cleaned_refunds cr
    LEFT JOIN fx_rates fr
        ON cr.currency = fr.base_ccy
        AND DATE(fr.date_utc) <= DATE(cr.created_at_utc)
    -- Select the most recent FX rate for each refund if multiple rates are available.
    QUALIFY row_number() OVER (
        PARTITION BY cr.id
        ORDER BY fr.date_utc DESC
    ) = 1
)
-- Final SELECT statement:
-- Aggregates the payment and refund data by date to calculate daily gross, refund, and net revenue.
SELECT
    DATE(pu.created_at_utc) AS date_utc,
    CAST(SUM(pu.gross_amount_eur) / 1.0 AS DECIMAL(18, 2)) AS gross_amount_eur,
    CAST(SUM(COALESCE(ru.refunds_eur, 0)) / 1.0 AS DECIMAL(18, 2)) AS refunds_eur,
    CAST((SUM(pu.gross_amount_eur) - SUM(COALESCE(ru.refunds_eur, 0))) / 1.0 AS DECIMAL(18, 2)) AS net_amount_eur,
    COUNT(pu.id) AS txn_count
FROM Payment_EUR pu
LEFT JOIN Refunds_EUR ru ON pu.id = ru.original_payment_id
GROUP BY 1
ORDER BY 1 DESC)

""")

In [ ]:
#Defined rule library for Sanity check
validation_rules = {
    "not_null_date": "SELECT COUNT(*) FROM fact_daily_revenue_eur WHERE date_utc IS NULL",
    "unique_dates": "SELECT COUNT(*) FROM (SELECT date_utc FROM fact_daily_revenue_eur GROUP BY 1 HAVING COUNT(*) > 1)",
    "net_refund_": "SELECT COUNT(*) FROM fact_daily_revenue_eur WHERE refunds_eur > gross_amount_eur",
    "net_revenue_calc": "SELECT COUNT(*) FROM fact_daily_revenue_eur WHERE ROUND(net_amount_eur, 2) != ROUND(gross_amount_eur - refunds_eur, 2)",
    #Checking reasoanleness for last 7 days
    "reasonableness_check": """
       SELECT COUNT(*)
FROM (
    SELECT
        net_amount_eur,
        LAG(net_amount_eur) OVER (ORDER BY date_utc) as prev_net,
        date_utc
    FROM fact_daily_revenue_eur
)
WHERE date_utc >= (SELECT MAX(date_utc) - INTERVAL 7 DAY FROM fact_daily_revenue_eur)
  AND ABS(net_amount_eur - prev_net) / NULLIF(prev_net, 0) > 0.5
    """
}

In [ ]:
def run_controls(rules, connection):
    """
    Executes validation rules for the transformed data and provides a summary report.
    Includes special handling for the reasonableness check, which reports as a warning/observation.
    """
    passed = 0
    results = []

    for name, query in rules.items():
        # Execute each validation query to get the error count
        error_count = connection.execute(query).fetchone()[0]

        if error_count == 0:
            passed += 1
            results.append(f" {name}")
        else:
            # handling for the 'reasonableness_check' to report it as an observation.
            if name == "reasonableness_check":
                results.append(f" {name}: {error_count} day(s) exceeded 50% change")
                passed += 1 # Treat reasonableness observations as 'passed' for the total count, but highlight them
            else:
                #For other rules, report as a failure with the count of violations found.
                results.append(f" {name}: {error_count} violations found")

    # Print the overall summary of the validation checks
    print(f"{len(rules)} checks run - {passed} passed")
    print("\n".join(results))

# Run the transformed data validation controls
run_controls(validation_rules, con)

5 checks run - 5 passed
 not_null_date
 unique_dates
 net_refund_
 net_revenue_calc
 reasonableness_check: 2 day(s) exceeded 50% change


In [ ]:
con.execute(""" SELECT * FROM fact_daily_revenue_eur ORDER BY date_utc DESC LIMIT 7""").fetch_df()

,date_utc,gross_amount_eur,refunds_eur,net_amount_eur,txn_count
0,2025-07-07,3461.3,0.0,3461.3,2
1,2025-07-06,1944.6,0.0,1944.6,2
2,2025-07-05,2400.0,0.0,2400.0,2
3,2025-07-04,1848.0,185.0,1663.0,1
4,2025-07-03,5105.0,924.0,4181.0,2
5,2025-07-02,3305.2,0.0,3305.2,2
6,2025-07-01,4255.0,1753.5,2501.5,2


In [ ]:
#Handled decimal mismatch with CSV file. It might be because of my local system can be ignored.
con.execute("""
COPY (
    SELECT
        date_utc,
        -- Convert to string to prevent Excel from auto-rounding
        PRINTF('%.2f', gross_amount_eur / 100.0) AS gross_amount_eur,
        PRINTF('%.2f', refunds_eur / 100.0) AS refunds_eur,
        PRINTF('%.2f', net_amount_eur / 100.0) AS net_amount_eur,
        txn_count
    FROM fact_daily_revenue_eur
    ORDER BY date_utc DESC
) TO 'fact_daily_revenue_eur.csv' (HEADER, DELIMITER ',');
""")

## Assumptions

*   **Payment Status Handling**: Only the latest record for payments with a 'posted' status is considered. Payments with other statuses (e.g., 'cancelled') are excluded, as are older versions of a payment if a more recent 'posted' update exists.
*   **1-Day FX Fallback**: When converting payment amounts to EUR, if an exact FX rate for the transaction date is not available, the most recent FX rate from up to one day prior to the transaction date is used as a fallback. If no rate within this window is found, the `gross_amount_eur` for that payment will be 0, and `missing_fx` will be true.

**NOTE- AS I HAVE USED COLAB TO COMPLETE THIS TASK SO I HAVE USED BUILT IN GEMINI FEATURE TO ADD COMMENTS (REVIEWED BY ME) AND FOR FEW SYNTAX CORRECTION TASKS. **